# Libraries

In [80]:
import pickle
import pandas as pd
pd.set_option('display.max_columns', None)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from pysr import PySRRegressor
import matplotlib.pyplot as plt
import joblib
import json
import itertools

[juliapkg] Found dependencies: c:\users\wanderlei_feng\documents\2025-2_andre_rimes\myenv-pysr\lib\site-packages\juliacall\juliapkg.json
[juliapkg] Found dependencies: c:\users\wanderlei_feng\documents\2025-2_andre_rimes\myenv-pysr\lib\site-packages\pysr\juliapkg.json
[juliapkg] Found dependencies: c:\users\wanderlei_feng\documents\2025-2_andre_rimes\myenv-pysr\lib\site-packages\juliapkg\juliapkg.json
[juliapkg] Locating Julia 1.10.3 - 1.11
[juliapkg] Querying Julia versions from https://julialang-s3.julialang.org/bin/versions.json
[juliapkg] WARNING: About to install Julia 1.11.8 to c:\Users\wanderlei_feng\Documents\2025-2_andre_rimes\myenv-pysr\julia_env\pyjuliapkg\install.
[juliapkg]   If you use juliapkg in more than one environment, you are likely to
[juliapkg]   have Julia installed in multiple locations. It is recommended to
[juliapkg]   install JuliaUp (https://github.com/JuliaLang/juliaup) or Julia
[juliapkg]   (https://julialang.org/downloads) yourself.
[juliapkg] Downloading

# Read dataset

In [81]:
filename = 'dataset_1.pkl'
with open(filename, 'rb') as f:
    dataset = pickle.load(f)

In [82]:
x = dataset[1]
x.head()

,diametro_da_armadura_mm,resistencia_do_aco_fy_mpa,numero_de_barras_em_uma_camada,espacamento_entre_as_barras_mm,resistencia_a_tracao_do_concreto_mpa,cobrimento_mm,distancia_ao_centro_de_gravidade_da_armadura_mm,largura_da_viga_b_mm,altura_da_viga_h_mm,area_de_aco_tracionada_as_mm2,taxa_de_armadura_conforme_mc2020,momento_fletor_ms_knm,fib_model_code_2010,eurocode_2,nbr_6118_wk1_fissuracao_estabilizada,nbr_6118_wk2_fissuracao_nao_estabilizada,clark_1956,desayiganesan_1985,oh_kang_1987,caldas_1997,gergely_e_lutz_1968,equacao_15_do_artigo_de_belarbi_e_hsu_1994,beta_conforme_fields_e_bischoff_2004,slip_de_yuan_et_al_2025,slip_de_biscaia_e_soares_2020
0,20.0,500,2,126.0,4.120341,25.0,39.0,216.0,201.0,628.0,0.0213,20.8,0.125734,0.125292,0.262468,0.157380,0.181006,0.312684,0.185969,0.129998,0.241528,0.149956,0.116566,0.109686,0.082007
1,20.0,500,2,126.0,4.120341,25.0,39.0,216.0,201.0,628.0,0.0213,27.3,0.176982,0.176359,0.344489,0.271111,0.244363,0.410398,0.256204,0.174746,0.317005,0.204348,0.197288,0.193326,0.156266
2,20.0,500,2,126.0,4.120341,25.0,39.0,216.0,201.0,628.0,0.0213,28.2,0.184078,0.183430,0.355846,0.289281,0.253135,0.423928,0.265928,0.180941,0.327456,0.212052,0.208041,0.205301,0.167096
3,20.0,500,2,126.0,4.120341,25.0,39.0,216.0,201.0,628.0,0.0213,32.6,0.218769,0.217999,0.411368,0.386596,0.296022,0.490072,0.313472,0.211232,0.378549,0.250139,0.259368,0.265222,0.221958
4,20.0,500,2,126.0,4.120341,25.0,39.0,216.0,201.0,628.0,0.0213,41.2,0.286574,0.285566,0.519888,0.617471,0.379848,0.619355,0.406398,0.270437,0.478411,0.311870,0.354810,0.388948,0.338382


In [83]:
y = dataset[2]
y.head()

,wexp_menos_wfib2020
0,0.073841
1,0.082873
2,0.084432
3,0.073162
4,0.072498


# Symbolic Regression

### Grid Search parameters

In [ ]:
n_folds = 10                                # Number of folds for cross-validation
size_test = 0.2                             # Proportion of the dataset to include in the test split
parsimony_levels = [0.001, 0.01]            # Regularization levels to control model complexity
max_size_levels = [20, 30]                  # Maximum size of the equations
pop_size_levels = [500, 500]                # Population size for the evolutionary algorithm
ncycles_per_iteration_levels = [1000]       # Number of cycles per iteration
niterations_levels = [500]                  # Number of iterations
weight_optimize_levels = [0.001]            # Weight optimization levels

In [85]:
combs_ = list(itertools.product(parsimony_levels, max_size_levels, pop_size_levels, ncycles_per_iteration_levels, niterations_levels, weight_optimize_levels))
da = pd.DataFrame(combs_, columns=['parsimony', 'max_size', 'pop_size', 'ncycles_per_iteration', 'niterations', 'weight_optimize'])
da

,parsimony,max_size,pop_size,ncycles_per_iteration,niterations,weight_optimize
0,0.001,20,500,1000,500,0.001
1,0.001,20,500,1000,500,0.001
2,0.001,30,500,1000,500,0.001
3,0.001,30,500,1000,500,0.001
4,0.010,20,500,1000,500,0.001
5,0.010,20,500,1000,500,0.001
6,0.010,30,500,1000,500,0.001
7,0.010,30,500,1000,500,0.001


### Train and Test

In [ ]:
dx = []
for j in range(n_folds):
   # Split the data in training and testing sets
   x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=size_test)
   # Standardize the features using StandardScaler z-score normalization
   scaler = StandardScaler()
   scaler.fit(x_train)
   x_train_scaled = scaler.transform(x_train)
   x_test_scaled = scaler.transform(x_test)
   
   for i, row in da.iterrows():
      # Save the scaler parameters
      dict_res = {}
      for i, col_name in enumerate(x_train.columns):
         dict_res[col_name] = {"avg": scaler.mean_[i], "std_dev": scaler.scale_[i]}
      
      # Train the PySR model
      model = PySRRegressor(
                              parsimony=row['parsimony'],           
                              maxsize=int(row['max_size']),
                              maxdepth=30,
                              population_size=int(row['pop_size']),
                              ncycles_per_iteration=int(row['ncycles_per_iteration']),
                              niterations=int(row['niterations']), 
                              adaptive_parsimony_scaling=50,
                              deterministic=False, 
                              should_optimize_constants=True,
                              optimizer_algorithm="BFGS",
                              optimizer_nrestarts=3,
                              binary_operators=["+", "-", "*", "/"],
                              unary_operators=["sin", "cos", "exp", "sqrt", "log", "abs", "square", "cube"],
                              weight_optimize=row['weight_optimize'],
                              verbosity=0,
                              progress=False,
                              parallelism='multithreading',
                              early_stop_condition=1e-8,
                           )
      model.fit(x_train_scaled, y_train)
      
      # Statistics and results
      y_pred_train = model.predict(x_train_scaled)
      y_pred_test = model.predict(x_test_scaled)
      rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
      rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
      r2_train = r2_score(y_train, y_pred_train)
      r2_test = r2_score(y_test, y_pred_test)
      print(f"Train RMSE: {rmse_train}")
      print(f"Test RMSE: {rmse_test}")
      print(f"Train R2: {r2_train}")
      print(f"Test R2: {r2_test}")
      best_eq = model.get_best()
      best_equation = best_eq['equation']
      print(f"Best Equation: {best_equation}")
      print("\n\n")
      dict_res['model'] = {
                              "train_rmse": rmse_train,
                              "test_rmse": rmse_test,
                              "train_r2": r2_train,
                              "test_r2": r2_test,
                              "best_equation": best_equation,
                              "scaler_params": scaler,
                              "model_object": model
                          }
      dx.append(dict_res)

### Save data

In [ ]:
aux = [{'metadados': 'index [1] data of model trained fold=0 grid=0, index [2] data of model trained fold=0 grid=1, ...'}]
results = aux + dx
with open(f'results_pysr_model_{filename}.pkl', 'wb') as f:
    pickle.dump(results, f)